# FLUX Unified Agent — ARC-AGI-3 Live Test

**Testing FLUX Unified Agent on REAL ARC-AGI-3 games** with the complete cognitive stack from `PHASE_UNIFIED_AGENT_SPEC.md`:

| Component | Purpose |
|-----------|--------|
| **FLUXUnifiedAgent** | Step-aware cognitive loop with VLM-ready architecture |
| **FrameDiffer** | Shows agent what changed after each action |
| **MovementStrategy** | Curiosity-driven navigation for movement games |
| **ClickStrategy** | Smart exploration for click games |
| **SpatialMemory** | Ice (curiosity) + Water (exploration mass) fields |
| **CausalTracker** | Action → effect learning |
| **RuleInducer** | Pattern → rule abstraction |

**Key Innovation:** The agent **SEES what happened** at each step via visual diff between frames, enabling:
1. Understanding the effect of the last action
2. Adapting strategy based on observed changes
3. Automatic handling of different game control types

---
## Cell 1: Environment Setup

In [ ]:
%%time
import os
import sys
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules

if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
else:
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")
print(f"Root: {ROOT}")
os.chdir(ROOT)

# Add all phase paths (including phase_unified for new agent)
phase_paths = [
    ROOT,
    ROOT / 'phases/phase1',
    ROOT / 'phases/phase2',
    ROOT / 'phases/phase8',
    ROOT / 'phases/phase8_8',
    ROOT / 'phases/phase9_arc',     # Spatial Memory
    ROOT / 'phases/phase_unified',  # Unified Agent (NEW)
]
for p in phase_paths:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"✓ Environment configured with {len(phase_paths)} paths")
print(f"✓ phase_unified path added for FLUXUnifiedAgent")

## Cell 2: Install Dependencies & Load API Key

In [ ]:
# Install arc-agi toolkit if not present
try:
    import arc_agi
    print("\u2713 arc-agi already installed")
except ImportError:
    print("Installing arc-agi...")
    !pip install -q arc-agi
    import arc_agi
    print("\u2713 arc-agi installed")

# ─────────────────────────────────────────────
# Load API Keys from Secrets (Kaggle/Colab first, then .env fallback)
# ─────────────────────────────────────────────

ARC_API_KEY = None
HF_TOKEN = None

# 1. Try Kaggle secrets first
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        ARC_API_KEY = secrets.get_secret('ARC_API_KEY')
        HF_TOKEN = secrets.get_secret('HF_TOKEN')
        if ARC_API_KEY:
            print("\u2713 ARC_API_KEY loaded from Kaggle secrets")
        if HF_TOKEN:
            print("\u2713 HF_TOKEN loaded from Kaggle secrets")
    except Exception as e:
        print(f"\u26a0 Kaggle secrets error: {e}")

# 2. Try Colab secrets
elif IN_COLAB:
    try:
        from google.colab import userdata
        ARC_API_KEY = userdata.get('ARC_API_KEY')
        HF_TOKEN = userdata.get('HF_TOKEN')
        if ARC_API_KEY:
            print("\u2713 ARC_API_KEY loaded from Colab secrets")
        if HF_TOKEN:
            print("\u2713 HF_TOKEN loaded from Colab secrets")
    except Exception as e:
        print(f"\u26a0 Colab secrets error: {e}")

# 3. Try environment variables
if not ARC_API_KEY:
    ARC_API_KEY = os.environ.get('ARC_API_KEY')
    if ARC_API_KEY:
        print("\u2713 ARC_API_KEY loaded from environment")

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("\u2713 HF_TOKEN loaded from environment")

# 4. Fallback to .env file (local development only)
if not ARC_API_KEY or not HF_TOKEN:
    env_file = ROOT / '.env'
    if env_file.exists():
        print("Loading from .env (local fallback)...")
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if not ARC_API_KEY and line.startswith('ARC_API_KEY='):
                    ARC_API_KEY = line.split('=', 1)[1]
                    print("\u2713 ARC_API_KEY loaded from .env")
                if not HF_TOKEN and line.startswith('hf_token='):
                    HF_TOKEN = line.split('=', 1)[1]
                    print("\u2713 HF_TOKEN loaded from .env")

# Set environment variables for libraries that need them
if ARC_API_KEY:
    os.environ['ARC_API_KEY'] = ARC_API_KEY
    print(f"  ARC_API_KEY: {ARC_API_KEY[:8]}...{ARC_API_KEY[-4:]}")
else:
    print("\u2717 ARC_API_KEY not found!")
    print("  Add to Kaggle/Colab secrets or set as environment variable")

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print(f"  HF_TOKEN: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
else:
    print("\u26a0 HF_TOKEN not found (model download may fail for private repos)")

## Cell 3: Device Setup & Load Model

In [ ]:
%%time
import torch
import numpy as np
from datetime import datetime

# Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load Flux-ARC.flx or download from HuggingFace
CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

model_path = CHECKPOINTS_DIR / 'Flux-ARC.flx'

if not model_path.exists():
    print("Downloading Flux-ARC.flx from HuggingFace...")
    from huggingface_hub import hf_hub_download
    
    try:
        hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename='checkpoints/Flux-ARC.flx',
            local_dir=str(ROOT),
            token=HF_TOKEN,  # Uses token from secrets loaded in Cell 2
        )
        print("\u2713 Downloaded Flux-ARC.flx")
    except Exception as e:
        print(f"\u26a0 Flux-ARC.flx download failed: {e}")
        print("  Trying Flux-Base.flx fallback...")
        try:
            hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename='checkpoints/Flux-Base.flx',
                local_dir=str(ROOT),
                token=HF_TOKEN,
            )
            model_path = CHECKPOINTS_DIR / 'Flux-Base.flx'
            print("\u2713 Downloaded Flux-Base.flx")
        except Exception as e2:
            print(f"\u2717 All downloads failed: {e2}")
            raise

# Load model
print(f"\nLoading {model_path.name}...")
flux_model = torch.load(str(model_path), map_location='cpu', weights_only=False)

print(f"\u2713 Loaded FLUX model")
print(f"  Format: {flux_model.get('format', 'unknown')}")
print(f"  Version: {flux_model.get('version', 'unknown')}")
print(f"  Components: {list(flux_model.get('components', {}).keys())[:8]}...")

## Cell 4: Initialize Cognitive Layer

In [ ]:
%%time
print("Initializing Cognitive Layer...")
print("=" * 60)

# Import cognitive components
from causal_tracker import CausalTracker
from rule_inducer import RuleInducer
from goal_planner import GoalPlanner, GoalType
from perception_field import PerceptionField
from arc_interface import GameAction, GameState, ActionCommand

# Initialize from model or fresh
if 'cognitive' in flux_model:
    print("Loading cognitive layer from checkpoint...")
    cog = flux_model['cognitive']
    
    ct_config = cog['causal_tracker']['config']
    causal_tracker = CausalTracker(
        max_history=ct_config.get('max_history', 1000),
        device='cpu',
    )
    causal_tracker.load_state_dict(cog['causal_tracker']['state_dict'])
    
    ri_config = cog['rule_inducer']['config']
    rule_inducer = RuleInducer(
        causal_tracker=causal_tracker,
        min_observations=ri_config.get('min_observations', 2),
        min_confidence=ri_config.get('min_confidence', 0.5),
    )
    rule_inducer.load_state_dict(cog['rule_inducer']['state_dict'])
    
    goal_planner = GoalPlanner()
    goal_planner.load_state_dict(cog['goal_planner']['state_dict'])
    
    pf_config = cog['perception_field']['config']
    perception_field = PerceptionField(
        max_size=pf_config.get('max_size', 64),
        fovea_radius=pf_config.get('fovea_radius', 5),
    )
    perception_field.load_state_dict(cog['perception_field']['state_dict'])
    
    print("\u2713 Loaded cognitive layer from Flux-ARC.flx")
else:
    print("Creating fresh cognitive layer...")
    causal_tracker = CausalTracker(max_history=1000, device='cpu')
    rule_inducer = RuleInducer(causal_tracker=causal_tracker)
    goal_planner = GoalPlanner()
    perception_field = PerceptionField(max_size=64, fovea_radius=5)
    print("\u2713 Created fresh cognitive layer")

print(f"\nCognitive Components Ready:")
print(f"  \u2713 CausalTracker (links: {len(causal_tracker.causal_links)})")
print(f"  \u2713 RuleInducer (rules: {len(rule_inducer.rules)})")
print(f"  \u2713 GoalPlanner")
print(f"  \u2713 PerceptionField (fovea_r={perception_field.fovea_radius})")

## Cell 5: Initialize Spatial Memory (Ice & Water)

In [ ]:
%%time
print("Initializing Spatial Memory (Ice & Water)...")
print("=" * 60)

from spatial_memory import SpatialMemory, NavigationTarget
import torch
import torch.nn.functional as F

# Create spatial memory instance
spatial_memory = SpatialMemory(
    max_size=64,          # ARC grids up to 64x64
    feature_dim=64,
    curiosity_threshold=0.1,
    device='cpu',
)

# ─────────────────────────────────────────────
# Monkey-patch encode_cell to handle colors >= 10
# ARC-AGI-3 may use extended color palette
# ─────────────────────────────────────────────
_original_encode_cell = spatial_memory.encode_cell

def _safe_encode_cell(self, color: int):
    """Safe encode_cell that clamps colors to 0-9."""
    safe_color = max(0, min(9, color))  # Clamp to valid range
    one_hot = F.one_hot(torch.tensor(safe_color), num_classes=10).float()
    one_hot = one_hot.to(self.device)
    return self.cell_encoder(one_hot)

# Bind the method
import types
spatial_memory.encode_cell = types.MethodType(_safe_encode_cell, spatial_memory)

print("✓ SpatialMemory initialized")
print("✓ Patched encode_cell to handle colors >= 10 (clamped to 0-9)")
print(f"  Max grid size: 64x64")
print(f"  Feature dim: 64")
print(f"  Curiosity threshold: 0.1")
print()
print("Dual-Field System:")
print("  ❄ CURIOSITY ICE — Highlights interesting locations")
print("  ≈ EXPLORATION MASS — Tracks visited locations")

## Cell 6: Initialize Grid Encoder (GridToWave)

In [ ]:
%%time
print("Initializing GridToWave Encoder...")
print("=" * 60)

from grid_adapters import GridToWave, WaveToGrid

WAVE_DIM = 432

# Initialize encoder with LARGER max_size for ARC-AGI-3 64x64 grids
# and num_colors=16 in case ARC-AGI-3 uses extended color palette
grid_to_wave = GridToWave(
    wave_dim=WAVE_DIM,
    max_size=64,      # ARC-AGI-3 uses 64x64 grids
    num_colors=16,    # Extended color range (0-15) for safety
    device='cpu'
)

# Try to load trained weights from model
if 'grid_adapters' in flux_model and 'encoder' in flux_model['grid_adapters']:
    try:
        grid_to_wave.load_state_dict(flux_model['grid_adapters']['encoder'], strict=False)
        print("✓ Loaded trained GridToWave from checkpoint (partial match)")
    except Exception as e:
        print(f"⚠ Could not load trained weights: {e}")
        print("  Using fresh GridToWave")
else:
    print("⚠ Using fresh GridToWave (no trained weights)")

grid_to_wave.eval()

# Test encoding with 64x64 grid
test_grid = [[0]*64 for _ in range(64)]
test_grid[32][32] = 5
with torch.no_grad():
    test_wave = grid_to_wave.encode(test_grid, mode='holistic')
    print(f"\nTest encoding: 64x64 grid -> [{WAVE_DIM}] wave")
    print(f"  Wave norm: {test_wave.norm().item():.4f}")

## Cell 7: Initialize FLUX Unified Agent

Using the new **FLUXUnifiedAgent** from `PHASE_UNIFIED_AGENT_SPEC.md` with:
- Step-aware cognitive loop (sees what changed)
- Control-specific strategies (movement/click)
- Frame differ for action → effect learning
- VLM integration ready (Qwen2.5-VL)

In [ ]:
%%time
print("Initializing FLUX Unified Agent...")
print("=" * 60)

# Import unified agent components
from unified_agent import FLUXUnifiedAgent, create_unified_agent
from frame_differ import FrameDiffer
from strategies import get_control_scheme, MovementStrategy, ClickStrategy
from game_loop import play_game_unified, GameResult

# Create the unified agent with spatial memory and causal tracker
agent = create_unified_agent(
    spatial_memory=spatial_memory,
    causal_tracker=causal_tracker,
    model=None,  # VLM optional - will use strategy fallback
    processor=None,
    device='cpu',
    verbose=True,
)

print("✓ FLUXUnifiedAgent created with:")
print(f"  - SpatialMemory: {spatial_memory is not None}")
print(f"  - CausalTracker: {causal_tracker is not None}")
print(f"  - VLM: {agent.vlm_available}")
print()
print("New capabilities (from PHASE_UNIFIED_AGENT_SPEC):")
print("  1. Step-aware cognitive loop")
print("     - Agent SEES what changed after each action")
print("     - Learns action → effect relationships")
print()
print("  2. Control-specific strategies")
print("     - MOVEMENT_ONLY: Curiosity-driven navigation")
print("     - CLICK_ONLY: Smart exploration, avoids repeat fails")
print("     - MOVE_AND_CLICK: Hybrid coordination")
print()
print("  3. Frame differ")
print("     - Computes visual diff between frames")
print("     - Tracks position changes, cell changes")
print()
print("  4. VLM-ready architecture")
print("     - Qwen2.5-VL integration prepared")
print("     - Falls back to strategy when no VLM")

## Cell 8: Test Connection to ARC API

In [ ]:
import requests

print("Testing ARC-AGI-3 API Connection...")
print("=" * 60)

ROOT_URL = "https://three.arcprize.org"

# Create session
session = requests.Session()
session.headers.update({
    "X-API-Key": ARC_API_KEY,
    "Accept": "application/json"
})

# Get games list
response = session.get(f"{ROOT_URL}/api/games")

if response.status_code == 200:
    games = response.json()
    print(f"\u2713 Connected! Found {len(games)} games")
    print(f"\nAvailable games:")
    for game in games[:10]:
        print(f"  - {game['game_id']}: {game.get('title', 'N/A')}")
    if len(games) > 10:
        print(f"  ... and {len(games) - 10} more")
else:
    print(f"\u2717 API Error: {response.status_code}")
    print(response.text)

## Cell 9: Play Single Game with FLUX Unified Agent

Using the new `play_game_unified()` from `game_loop.py` which:
- Uses step-aware cognitive loop
- Shows agent what changed after each action
- Selects strategy based on control scheme

In [ ]:
def play_game_with_flux(game_id: str, max_actions: int = 100, verbose: bool = True):
    """
    Play a single ARC-AGI-3 game using FLUX Unified Agent.
    
    Uses the new step-aware cognitive loop from PHASE_UNIFIED_AGENT_SPEC:
    - Agent sees what changed after each action
    - Selects strategy based on control scheme
    - Learns action → effect relationships
    """
    print(f"\n{'='*60}")
    print(f"Playing: {game_id} with FLUX Unified Agent")
    print(f"{'='*60}")
    
    # Reset agent (clears all state for fresh game)
    agent.reset()
    agent.verbose = verbose
    
    # Open scorecard
    response = session.post(
        f"{ROOT_URL}/api/scorecard/open",
        json={"tags": ["flux-unified", game_id]}
    )
    if response.status_code != 200:
        print(f"✗ Failed to open scorecard: {response.text}")
        return None
    
    card_id = response.json()["card_id"]
    print(f"Scorecard: {card_id}")
    
    # Reset game
    response = session.post(
        f"{ROOT_URL}/api/cmd/RESET",
        json={"game_id": game_id, "card_id": card_id}
    )
    if response.status_code != 200:
        print(f"✗ RESET failed: {response.text}")
        return None
    
    game_data = response.json()
    guid = game_data["guid"]
    state = game_data["state"]
    score = game_data.get("score", 0)
    frame = game_data.get("frame", [[]])
    available_actions = game_data.get("available_actions", [1, 2, 3, 4, 5])
    
    # Convert to list of ints if needed
    if available_actions and hasattr(available_actions[0], 'value'):
        available_actions = [a.value for a in available_actions]
    
    # Initialize unified agent for this game
    agent.start_game(game_id, frame, available_actions)
    
    print(f"Initial state: {state}, Score: {score}")
    print(f"Control scheme: {agent.control_scheme}")
    print(f"Available actions: {available_actions}")
    
    # Play loop using step-aware cognitive loop
    actions_taken = 0
    level_wins = 0
    last_action = None
    
    while state == "NOT_FINISHED" and actions_taken < max_actions:
        # Update available actions
        if available_actions and hasattr(available_actions[0], 'value'):
            available_actions = [a.value for a in available_actions]
        
        try:
            # Agent step (step-aware cognitive loop)
            action, action_data, reasoning = agent.step(
                frame=frame,
                available_actions=available_actions,
                last_action=last_action,
            )
            
            action_name = ['RESET', 'UP', 'DOWN', 'LEFT', 'RIGHT', 'INTERACT', 'CLICK', 'UNDO'][action] if action < 8 else f'ACTION{action}'
            
            if verbose and actions_taken % 10 == 0:
                print(f"\nStep {actions_taken}: action={action_name}, pos={agent.current_position}")
            
        except Exception as e:
            print(f"✗ Agent error at step {actions_taken}: {e}")
            import traceback
            traceback.print_exc()
            break
        
        # Build request
        request_data = {
            "game_id": game_id,
            "card_id": card_id,
            "guid": guid,
        }
        request_data.update(action_data)
        
        # Take action
        action_endpoint = f"ACTION{action}" if action > 0 else "RESET"
        response = session.post(
            f"{ROOT_URL}/api/cmd/{action_endpoint}",
            json=request_data
        )
        
        if response.status_code != 200:
            print(f"✗ Action failed: {response.text}")
            break
        
        game_data = response.json()
        state = game_data["state"]
        new_score = game_data.get("score", score)
        frame = game_data.get("frame", frame)
        new_available = game_data.get("available_actions", available_actions)
        
        if new_available and hasattr(new_available[0], 'value'):
            available_actions = [a.value for a in new_available]
        else:
            available_actions = new_available
        
        # Track level wins
        if new_score > score:
            print(f"  ↑ Level completed! Score: {score} -> {new_score}")
            level_wins += 1
        
        score = new_score
        last_action = action
        actions_taken += 1
    
    # Close scorecard
    try:
        session.post(f"{ROOT_URL}/api/scorecard/close", json={"card_id": card_id})
    except:
        pass
    
    # Capture spatial memory snapshot
    grid_size = 64
    if agent.last_grid:
        grid_size = max(len(agent.last_grid), len(agent.last_grid[0]) if agent.last_grid else 64)
    
    spatial_snapshot = {
        'exploration_mass': spatial_memory.exploration_mass[:grid_size, :grid_size].clone().cpu().numpy(),
        'curiosity_field': spatial_memory.curiosity_field[:grid_size, :grid_size].clone().cpu().numpy(),
        'grid_size': grid_size,
        'final_pos': agent.current_position,
    }
    
    # Results
    result = {
        'game_id': game_id,
        'card_id': card_id,
        'final_state': state,
        'final_score': score,
        'actions_taken': actions_taken,
        'level_wins': level_wins,
        'agent_stats': agent.get_stats(),
        'spatial_snapshot': spatial_snapshot,
    }
    
    print(f"\n{'-'*40}")
    print(f"Result: {state}")
    print(f"Score: {score}")
    print(f"Actions: {actions_taken}")
    print(f"Levels won: {level_wins}")
    print(f"Effectiveness: {result['agent_stats'].get('effectiveness_rate', 0):.1%}")
    print(f"Scorecard: {ROOT_URL}/scorecards/{card_id}")
    
    return result

print("✓ play_game_with_flux() defined")
print("  Uses step-aware cognitive loop from PHASE_UNIFIED_AGENT_SPEC")

## Cell 10: Run FLUX on ls20

In [ ]:
%%time
# Test on ls20 - Agent Reasoning game
result_ls20 = play_game_with_flux("ls20", max_actions=50, verbose=True)

## Cell 11: Run FLUX on ft09

In [ ]:
%%time
# Test on ft09 - Elementary Logic game
result_ft09 = play_game_with_flux("ft09", max_actions=50, verbose=True)

## Cell 12: Run FLUX on vc33

In [ ]:
%%time
# Test on vc33 - Orchestration game
result_vc33 = play_game_with_flux("vc33", max_actions=50, verbose=True)

## Cell 13: Aggregate Results

In [ ]:
print("\n" + "=" * 60)
print("FLUX UNIFIED AGENT — ARC-AGI-3 TEST RESULTS")
print("=" * 60)

results = [r for r in [result_ls20, result_ft09, result_vc33] if r is not None]

print(f"\nGames tested: {len(results)}")
print(f"\n{'Game':<10} {'Scheme':<18} {'State':<12} {'Score':<8} {'Actions':<8} {'Effect%':<8}")
print("-" * 70)

total_score = 0
total_actions = 0
total_wins = 0

for r in results:
    scheme = r['agent_stats'].get('control_scheme', 'N/A')
    eff_rate = r['agent_stats'].get('effectiveness_rate', 0)
    print(f"{r['game_id']:<10} {scheme:<18} {r['final_state']:<12} {r['final_score']:<8.1f} {r['actions_taken']:<8} {eff_rate:<8.1%}")
    total_score += r['final_score']
    total_actions += r['actions_taken']
    total_wins += r['level_wins']

print("-" * 70)
avg_eff = sum(r['agent_stats'].get('effectiveness_rate', 0) for r in results) / max(1, len(results))
print(f"{'TOTAL':<10} {'':<18} {'':<12} {total_score:<8.1f} {total_actions:<8} {avg_eff:<8.1%}")

print(f"\n\nStep-Aware Cognitive Stats:")
print(f"  Control schemes used: {set(r['agent_stats'].get('control_scheme') for r in results)}")
print(f"  Average effectiveness: {avg_eff:.1%}")
print(f"  Causal links learned: {len(causal_tracker.causal_links)}")
print(f"  Rules induced: {len(rule_inducer.rules)}")

# Check against spec targets
print(f"\n\nSpec Targets:")
print(f"  ls20: {next((r['final_score'] for r in results if r['game_id']=='ls20'), 0):.1f}/0.3")
print(f"  ft09: {next((r['final_score'] for r in results if r['game_id']=='ft09'), 0):.1f}/0.3")
print(f"  vc33: {next((r['final_score'] for r in results if r['game_id']=='vc33'), 0):.1f}/0.3")

## Cell 14: Visualize Spatial Memory

In [ ]:
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# Visualize Spatial Memory for EACH GAME separately
# ─────────────────────────────────────────────

results = [r for r in [result_ls20, result_ft09, result_vc33] if r is not None]

if not results:
    print("No results to visualize!")
else:
    n_games = len(results)
    
    # Create figure with 2 rows per game (mass + ice)
    fig, axes = plt.subplots(n_games, 2, figsize=(14, 5 * n_games))
    
    # Handle single game case (axes won't be 2D)
    if n_games == 1:
        axes = [axes]
    
    for i, result in enumerate(results):
        game_id = result['game_id']
        snap = result.get('spatial_snapshot', {})
        
        mass = snap.get('exploration_mass')
        ice = snap.get('curiosity_field')
        grid_size = snap.get('grid_size', 64)
        final_pos = snap.get('final_pos', (0, 0))
        detected_pos = snap.get('detected_agent_pos')
        clicks = snap.get('clicks_explored', 0)
        
        if mass is None or ice is None:
            print(f"[{game_id}] No spatial data captured")
            continue
        
        # Exploration Mass (Water)
        im1 = axes[i][0].imshow(mass, cmap='hot', vmin=0)
        axes[i][0].set_title(f'{game_id}: Exploration Mass (Water)\nWhere FLUX explored')
        axes[i][0].set_xlabel(f'Final pos: {final_pos} | Detected: {detected_pos}')
        plt.colorbar(im1, ax=axes[i][0])
        
        # Mark final position
        if final_pos:
            axes[i][0].scatter([final_pos[1]], [final_pos[0]], c='cyan', s=100, marker='*', label='Final')
        
        # Curiosity Ice
        im2 = axes[i][1].imshow(ice, cmap='cool', vmin=0)
        axes[i][1].set_title(f'{game_id}: Curiosity Ice\nWhat was interesting')
        axes[i][1].set_xlabel(f'Score: {result["final_score"]} | Actions: {result["actions_taken"]} | Clicks: {clicks}')
        plt.colorbar(im2, ax=axes[i][1])
    
    plt.tight_layout()
    plt.show()
    
    # ─────────────────────────────────────────────
    # ASCII visualizations for each game
    # ─────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("SPATIAL MEMORY ASCII VIEW — Per Game")
    print("=" * 80)
    
    for result in results:
        game_id = result['game_id']
        snap = result.get('spatial_snapshot', {})
        grid_size = snap.get('grid_size', 64)
        final_pos = snap.get('final_pos', (0, 0))
        mass = snap.get('exploration_mass')
        ice = snap.get('curiosity_field')
        
        print(f"\n{'─' * 60}")
        print(f"Game: {game_id}")
        print(f"Score: {result['final_score']} | Actions: {result['actions_taken']} | Final Position: {final_pos}")
        print(f"{'─' * 60}")
        
        if mass is None:
            print("  (No spatial data)")
            continue
        
        # Simple ASCII visualization
        h, w = mass.shape
        display_h = min(h, 32)  # Limit for display
        display_w = min(w, 64)
        
        print()
        for r in range(display_h):
            row_str = ""
            for c in range(display_w):
                m = mass[r, c]
                i = ice[r, c]
                
                if (r, c) == final_pos:
                    row_str += "@ "
                elif m > 10:
                    row_str += "● "  # Heavy exploration
                elif m > 1:
                    row_str += "· "  # Light exploration
                elif i > 10:
                    row_str += "🧊"  # High curiosity
                elif i > 1:
                    row_str += "❄ "  # Low curiosity
                else:
                    row_str += "  "  # Unexplored
            print(row_str)
        
        if h > display_h or w > display_w:
            print(f"  ... (showing {display_h}x{display_w} of {h}x{w})")
        
        print(f"\nLegend: @ final pos | ● explored | · visited | 🧊 high ice | ❄ ice")
        print(f"Stats: {snap.get('clicks_explored', 0)} cells clicked")

## Cell 15: Summary

In [ ]:
print("=" * 60)
print("FLUX UNIFIED AGENT — ARC-AGI-3 LIVE TEST COMPLETE")
print("=" * 60)

print(f"""
Model: {model_path.name}
Agent: FLUXUnifiedAgent (PHASE_UNIFIED_AGENT_SPEC v1.0)
Games tested: {len(results)}
Total levels won: {total_wins}
Total actions: {total_actions}

Step-Aware Cognitive Stack:
  ✓ FrameDiffer (sees what changed)
  ✓ Control-specific strategies
      - MovementStrategy (for ls20-style games)
      - ClickStrategy (for ft09/vc33-style games)
  ✓ SpatialMemory (Ice & Water navigation)
  ✓ CausalTracker ({len(causal_tracker.causal_links)} links learned)
  ✓ RuleInducer ({len(rule_inducer.rules)} rules)
  ✓ GridToWave encoder

Agent Stats:
""")

if results:
    stats = results[-1]['agent_stats']
    for key, val in stats.items():
        if isinstance(val, float):
            print(f"    {key}: {val:.3f}")
        else:
            print(f"    {key}: {val}")

print(f"""
View scorecards at:
""")

for r in results:
    print(f"  {r['game_id']}: {ROOT_URL}/scorecards/{r['card_id']}")

print(f"""

Success Criteria (from spec):
  Target: ls20 >0.3, ft09 >0.3, vc33 >0.3
  Actual: {', '.join(f"{r['game_id']}={r['final_score']}" for r in results)}

Next Steps:
  1. Add Qwen2.5-VL for visual reasoning
  2. Improve rule induction from causal observations
  3. Test on more games (wa30, tr87, sc25, etc.)
  4. Submit to competition (OperationMode.COMPETITION)
""")